In [27]:
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.utils import read_matrix_from_csv

from scanpy import pl, tl
from scipy.sparse import hstack

import anndata

In [6]:
ann = anndata.io.read_h5ad("D:/Data/nichenetpy/annData/annData3531889.h5")
ann

AnnData object with n_obs × n_vars = 5027 × 13541
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nGene', 'nUMI', 'aggregate', 'res.0.6', 'celltype'
    layers: 'counts', 'data', 'scale.data'

In [3]:
celltype_counts = ann.obs["celltype"].value_counts()
celltype_counts

celltype
CD4 T    2562
CD8 T    1645
B         382
Treg      199
NK        131
Mono       90
DC         18
Name: count, dtype: int64

In [8]:
receiver = "CD8 T"
cells_oi = list(ann.obs.loc[ann.obs["celltype"] == receiver].index)
cells_oi[:10]

['W380370',
 'W380378',
 'W380386',
 'W380393',
 'W380395',
 'W380401',
 'W380410',
 'W380417',
 'W380418',
 'W380422']

In [69]:
mat = ann.layers["data"].T

In [70]:
col2index = dict(zip(ann.obs.index, range(len(ann.obs.index))))
ids = [col2index[name] for name in cells_oi]
exprs_m = hstack([mat[:, id] for id in ids])
nrows, ncols = exprs_m.get_shape()
print(nrows, ncols)

13541 1645


In [71]:
for i in range(len(exprs_m.data)):
    exprs_m.data[i] = 1

In [72]:
pct = 0.05
genes = list((gene, val) for gene, val in enumerate(exprs_m.sum(axis=1)/ncols) if val > pct)
genes[:10]

[(1, matrix([[0.05410334]])),
 (2, matrix([[0.07537994]])),
 (5, matrix([[0.05106383]])),
 (7, matrix([[0.1775076]])),
 (11, matrix([[0.07355623]])),
 (15, matrix([[0.07051672]])),
 (17, matrix([[0.07112462]])),
 (19, matrix([[0.05227964]])),
 (20, matrix([[0.05957447]])),
 (24, matrix([[0.10091185]]))]

In [73]:
# expected: 3903
print(len(genes))

3903


In [140]:
predictor = LigandActivityPredictor(*read_matrix_from_csv("D:/Data/nichenetpy/testargs/ligand_target_matrix.csv"))

In [5]:
ligand_activities = predictor.predict_ligand_activities(
    geneset,
    background_expressed_genes,
    potential_ligands
)
ligand_activities = sorted(ligand_activities.items(), key=lambda x : x[1]["aupr_corrected"], reverse=True)
ligand_activities[:10]

[('"Ifna1"',
  {'aupr': np.float64(0.4178628711859948),
   'aupr_corrected': np.float64(0.3423379573658757)}),
 ('"Ifnl3"',
  {'aupr': np.float64(0.391939644541973),
   'aupr_corrected': np.float64(0.3164147307218539)}),
 ('"Ifnb1"',
  {'aupr': np.float64(0.38720798251986854),
   'aupr_corrected': np.float64(0.3116830686997495)}),
 ('"Il27"',
  {'aupr': np.float64(0.37815297574454027),
   'aupr_corrected': np.float64(0.30262806192442115)}),
 ('"Ifng"',
  {'aupr': np.float64(0.3655665422088432),
   'aupr_corrected': np.float64(0.29004162838872416)}),
 ('"Ifnk"',
  {'aupr': np.float64(0.27522591434367766),
   'aupr_corrected': np.float64(0.19970100052355858)}),
 ('"Ifne"',
  {'aupr': np.float64(0.27370092284997777),
   'aupr_corrected': np.float64(0.19817600902985869)}),
 ('"Ebi3"',
  {'aupr': np.float64(0.2545082226394273),
   'aupr_corrected': np.float64(0.1789833088193082)}),
 ('"Ifnl2"',
  {'aupr': np.float64(0.24461164764885127),
   'aupr_corrected': np.float64(0.16908673382873218)}